In [1]:
import sys
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time

# 1. Chemins
root_path = os.path.abspath("../..")
if root_path not in sys.path:
    sys.path.append(root_path)

from GLC.data_loading.pytorch_dataset import GeoLifeCLEF2021Dataset

BASE_PATH = Path(r"C:\Users\abdou\Downloads\geolifeclef-2021")
DATA_PATH = BASE_PATH / "data"
PATCHES_PATH = DATA_PATH / "patches_sample"

# 2. Appareil (CPU)
device = torch.device("cpu")

# 3. Chargement et filtrage (pour ne garder que l'échantillon)
train_dataset = GeoLifeCLEF2021Dataset(root=DATA_PATH, subset="train", use_rasters=False)
val_dataset = GeoLifeCLEF2021Dataset(root=DATA_PATH, subset="val", use_rasters=False)

valid_files = list(PATCHES_PATH.rglob("*_rgb.jpg"))
valid_ids = set([int(f.stem.split('_')[0]) for f in valid_files])

# Filtre Train
train_indices = [i for i, obs_id in enumerate(train_dataset.observation_ids) if obs_id in valid_ids]
train_dataset.observation_ids = train_dataset.observation_ids[train_indices]
train_dataset.coordinates = train_dataset.coordinates[train_indices]
if train_dataset.targets is not None: train_dataset.targets = train_dataset.targets[train_indices]

# Filtre Val
val_indices = [i for i, obs_id in enumerate(val_dataset.observation_ids) if obs_id in valid_ids]
val_dataset.observation_ids = val_dataset.observation_ids[val_indices]
val_dataset.coordinates = val_dataset.coordinates[val_indices]
if val_dataset.targets is not None: val_dataset.targets = val_dataset.targets[val_indices]

# Dataloaders et Classes
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
vrai_num_classes = int(train_dataset.targets.max()) + 1

print("Cellule 1 terminée. Données prêtes !")

Cellule 1 terminée. Données prêtes !


In [2]:
# Modèle 1 : Unimodal
class SimpleGeoLifeCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleGeoLifeCNN, self).__init__()
        self.conv1 = nn.Conv2d(6, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 64 * 64, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) 
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# Modèle 2 : Bimodal
class BimodalGeoLifeCNN(nn.Module):
    def __init__(self, num_classes):
        super(BimodalGeoLifeCNN, self).__init__()
        self.pool = nn.MaxPool2d(2, 2)
        self.conv1_vis = nn.Conv2d(4, 16, kernel_size=3, padding=1)
        self.conv2_vis = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3_vis = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv1_topo = nn.Conv2d(2, 16, kernel_size=3, padding=1)
        self.conv2_topo = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3_topo = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64*32*32 + 32*32*32, 512)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x_vis, x_topo = x[:, :4, :, :], x[:, 4:, :, :]
        out_vis = self.pool(F.relu(self.conv3_vis(self.pool(F.relu(self.conv2_vis(self.pool(F.relu(self.conv1_vis(x_vis)))))))))
        out_topo = self.pool(F.relu(self.conv3_topo(self.pool(F.relu(self.conv2_topo(self.pool(F.relu(self.conv1_topo(x_topo)))))))))
        out = torch.cat((out_vis.view(out_vis.size(0), -1), out_topo.view(out_topo.size(0), -1)), dim=1)
        return self.fc2(self.dropout(F.relu(self.fc1(out))))

# La fonction qui retourne l'historique
def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, num_epochs=2):
    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # Train
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)
            
        history['train_loss'].append(train_loss / len(train_loader.dataset))
        
        # Eval
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                correct += (torch.max(outputs.data, 1)[1] == labels).sum().item()
                total += labels.size(0)
                
        history['val_loss'].append(val_loss / len(val_loader.dataset))
        history['val_accuracy'].append(100 * correct / total)
        print(f"Epoque {epoch+1}/{num_epochs} [{time.time() - start_time:.1f}s] - Val Loss: {history['val_loss'][-1]:.4f}")
        
    return history

print("✅ Cellule 2 terminée. Architectures définies !")

✅ Cellule 2 terminée. Architectures définies !


In [ ]:
num_epochs = 3 # Tu peux mettre 3 si tu veux
criterion = nn.CrossEntropyLoss()

print("🥊 ROUND 1 : Entraînement du CNN Unimodal...")
model_uni = SimpleGeoLifeCNN(num_classes=vrai_num_classes).to(device)
optimizer_uni = optim.Adam(model_uni.parameters(), lr=0.001)
history_uni = train_and_evaluate(model_uni, train_loader, val_loader, criterion, optimizer_uni, num_epochs)

print("\n🥊 ROUND 2 : Entraînement du CNN Bimodal...")
model_bi = BimodalGeoLifeCNN(num_classes=vrai_num_classes).to(device)
optimizer_bi = optim.Adam(model_bi.parameters(), lr=0.001)
history_bi = train_and_evaluate(model_bi, train_loader, val_loader, criterion, optimizer_bi, num_epochs)

print("✅ Cellule 3 terminée. Les historiques sont sauvegardés en mémoire !")

🥊 ROUND 1 : Entraînement du CNN Unimodal...
Epoque 1/3 [2261.4s] - Val Loss: 8.9369
Epoque 2/3 [2281.2s] - Val Loss: 9.0411
Epoque 3/3 [2327.6s] - Val Loss: 9.3789

🥊 ROUND 2 : Entraînement du CNN Bimodal...


In [ ]:
MODELS_PATH = BASE_PATH / "models"
os.makedirs(MODELS_PATH, exist_ok=True)

torch.save(model_uni.state_dict(), MODELS_PATH / "cnn_unimodal.pth")
torch.save(model_bi.state_dict(), MODELS_PATH / "cnn_bimodal.pth")
print("💾 Modèles sauvegardés avec succès sur le disque !")

In [ ]:
%matplotlib inline
epochs_range = range(1, num_epochs + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(epochs_range, history_uni['val_loss'], label='Unimodal Loss', marker='o', linewidth=2)
ax1.plot(epochs_range, history_bi['val_loss'], label='Bimodal Loss', marker='s', linewidth=2)
ax1.set_title('Perte (Plus bas c\'est mieux)')
ax1.legend()
ax1.grid(True, linestyle='--')

ax2.plot(epochs_range, history_uni['val_accuracy'], label='Unimodal Acc', marker='o', linewidth=2)
ax2.plot(epochs_range, history_bi['val_accuracy'], label='Bimodal Acc', marker='s', linewidth=2)
ax2.set_title('Précision (Plus haut c\'est mieux)')
ax2.legend()
ax2.grid(True, linestyle='--')

plt.tight_layout()
plt.show()